# Week 2 개념 정리 — LLM 내부 구조, 오디오 AI, 샌드박싱

Day 1-4를 위한 실행 가능한 동반 노트북이며, 일별 학습 노트와 짝을 맞춰 날짜별로 구성했습니다.
모든 코드 셀에는 각 단계가 *왜* 필요한지 설명하는 상세한 주석이 달려 있고, 텐서/배열을
만들거나 변형하는 모든 줄에는 명시적인 shape 주석이 붙어 있습니다.

**이 샌드박스에서 실제로 실행된 것과 그렇지 않은 것:** `tiktoken` 기반 토큰화 예제, 처음부터
직접 구현한 모든 `numpy`/`scipy` 코드(BPE, 셀프 어텐션, KV 캐시 메모리 시뮬레이션, 로그-멜
스펙트로그램 파이프라인, 검색 평가 지표, RLIMIT_AS 플랫폼 실패 케이스를 포함한
`subprocess`/`resource` 샌드박싱 데모), TF-IDF 기반 바이 인코더 실패 사례 데모, 그리고
`pyttsx3` OS 레벨 TTS 호출은 모두 이 환경에서 직접 실행해 실제 출력을 검증했고, 주변
마크다운 셀에 그 결과를 그대로 옮겨 놓았습니다. `sentence-transformers`(`SentenceTransformer`,
`CrossEncoder`), `whisper`, 호스팅형 `Sandbox` SDK를 호출하는 셀은 **이 환경에서 실행하지
않았습니다** — GPU 없음, torch 없음, 수백 MB짜리 모델 다운로드, 실제 샌드박스 제공업체
자격 증명 모두 이 환경에서는 현실적이지 않습니다 — 하지만 실제 최신 API 그대로 작성했고
아래에 명확히 표시해 두었습니다.

## Day 1: 토큰화, 어텐션, KV 캐시

네 부분으로 구성되며, 모두 직접 실행해 검증했습니다: 처음부터 구현한 BPE 트레이너, 영어와
한국어에 대한 실제 `tiktoken` 토큰 수, 처음부터 구현한 numpy 셀프 어텐션(코잘 마스킹 포함),
그리고 실제 70억 파라미터급 모델 규모의 숫자를 사용한 KV 캐시 메모리 증가 시뮬레이션.

In [ ]:
# --- 작은 장난감 코퍼스에 대해 처음부터 구현한 BPE 트레이너 ---
# Sennrich 등의 원조 BPE 논문과 같은 예시: 단어 4개, 각각을 문자 단위로 쪼개고 단어 끝
# 마커 '_'를 붙임(그래야 단어 경계의 'est'와 단어 중간의 'est'가, 같은 방식으로
# 동작한다는 게 증명되기 전까지는 서로 다른 심볼로 취급됨).
import collections

corpus = {"low": 5, "lower": 2, "newest": 6, "widest": 3}  # 단어 -> 빈도

def word_to_symbols(word):
    return list(word) + ["_"]

# vocab: 심볼 튜플 -> 빈도, 예: ('l', 'o', 'w', '_') -> 5
vocab = {tuple(word_to_symbols(w)): f for w, f in corpus.items()}

def get_pair_counts(vocab):
    # vocab 전체에서 인접한 심볼 쌍을, 단어 빈도로 가중해 카운트
    pairs = collections.Counter()
    for symbols, freq in vocab.items():
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    # `pair`의 모든 등장을 하나의 병합된 심볼로 교체
    a, b = pair
    merged = a + b
    new_vocab = {}
    for symbols, freq in vocab.items():
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(merged)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        new_vocab[tuple(new_symbols)] = new_vocab.get(tuple(new_symbols), 0) + freq
    return new_vocab

print("초기 vocab (단어 -> 심볼 시퀀스):")
for symbols, freq in vocab.items():
    print(f"  {symbols}  freq={freq}")

num_merges = 6
for step in range(num_merges):
    pairs = get_pair_counts(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)  # 가장 빈번한 인접 쌍 -> 다음 병합 규칙
    vocab = merge_vocab(best, vocab)
    print(f"\nstep {step + 1}: merge {best} (seen {pairs[best]}x) -> '{best[0] + best[1]}'")
    for symbols, freq in vocab.items():
        print(f"  {symbols}  freq={freq}")
# 예상되는(검증된) 병합 순서: ('e','s') -> ('es','t') -> ('est','_') -> ('l','o') -> ('lo','w') -> ('n','e')

In [ ]:
# --- 실제 tiktoken 토큰 수: 영어 vs. 한국어, 거의 같은 의미 ---
# `pip install tiktoken` 필요(순수 Python에 가까워 수 초 만에 설치되고 GPU도 필요 없음).
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 계열의 바이트 레벨 BPE 토크나이저
en = "The quarterly report is ready for review."
ko = "분기 보고서 검토 준비가 완료되었습니다."

en_ids = enc.encode(en)  # -> list[int]
ko_ids = enc.encode(ko)  # -> list[int]
print(len(en_ids), "tokens for English:", en_ids)
print(len(ko_ids), "tokens for Korean:", ko_ids)
print(f"chars/token  EN={len(en)/len(en_ids):.2f}  KO={len(ko)/len(ko_ids):.2f}")
# 실제 검증된 출력: 영어 8토큰 vs. 한국어 20토큰 -- 거의 같은 내용인데 토큰 수는 2.5배.
# 병합 어휘가 대부분 영어 중심 학습 텍스트로부터 학습됐기 때문.

# 각 한국어 토큰을 다시 원시 바이트로 디코딩해서 *왜* 그런지 확인: 일부 음절은 자기만의
# 병합 토큰을 얻지 못하고 개별 UTF-8 바이트까지 완전히 쪼개짐.
print("\n한국어 토큰 -> 원시 바이트:")
for tid in ko_ids:
    b = enc.decode_single_token_bytes(tid)
    print(f"  id={tid:>6}  bytes={b!r}  ({len(b)} byte{'s' if len(b) != 1 else ''})")
# 검증됨: '검토'의 '토'는 하나의 토큰이 아니라 -- 서로 다른 단일 바이트 토큰 세 개
# (b'\xed', b'\x86', b'\xa0', 그 한 글자를 이루는 원시 UTF-8 3바이트)로 쪼개짐.

In [ ]:
# --- 처음부터 구현한 스케일드 닷프로덕트 셀프 어텐션, 매 단계마다 shape 표기 ---
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)  # 수치 안정성을 위함, 결과 자체는 바뀌지 않음
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

np.random.seed(0)
tokens = ["The", "cat", "sat", "on", "mat"]
seq_len, d_model = len(tokens), 8  # 실제 70억 파라미터 모델은 d_model=4096; 8은 출력 숫자를 읽기 쉽게 유지

X = np.random.randn(seq_len, d_model)  # shape: (5, 8) -- 토큰 임베딩 + 위치 정보, 이미 합산됨

# 실제 모델에서는 학습됨; 여기서는 무작위지만 고정값으로, 학습 없이 메커니즘만 확인
W_q = np.random.randn(d_model, d_model) * 0.1  # shape: (8, 8)
W_k = np.random.randn(d_model, d_model) * 0.1
W_v = np.random.randn(d_model, d_model) * 0.1

Q = X @ W_q  # shape: (5, 8) -- "각 토큰이 무엇을 찾고 있는가"
K = X @ W_k  # shape: (5, 8) -- "각 토큰이 자신에 대해 무엇을 내세우는가"
V = X @ W_v  # shape: (5, 8) -- "주목받았을 때 각 토큰이 제공하는 내용"

scores = (Q @ K.T) / np.sqrt(d_model)  # shape: (5, 5) -- scores[i, j] = query_i . key_j, 스케일링됨
weights = softmax(scores, axis=-1)     # shape: (5, 5), 각 행의 합이 1.0 -- 어텐션 가중치
context = weights @ V                  # shape: (5, 8) -- 토큰별 새로운, 문맥이 섞인 표현

print("어텐션 가중치 (행=쿼리 토큰, 열=키 토큰):")
print(np.round(weights, 2))
print("행의 합 (모두 1.0이어야 함):", weights.sum(axis=1))
print("context shape:", context.shape)

In [ ]:
# --- 코잘 마스킹: 토큰 i는 i 이하의 토큰에만 주목할 수 있음 (생성에 필요) ---
causal_mask = np.triu(np.ones((seq_len, seq_len)), k=1).astype(bool)  # shape: (5, 5), 대각선 위쪽만 True
masked_scores = np.where(causal_mask, -np.inf, scores)  # 미래 위치는 -inf 점수를 받음
causal_weights = softmax(masked_scores, axis=-1)        # shape: (5, 5) -- 위쪽 삼각형은 정확히 0으로 사라짐

print("코잘 어텐션 가중치 (위쪽 삼각형은 반드시 0이어야 함):")
print(np.round(causal_weights, 2))

In [ ]:
# --- KV 캐시 메모리 사용량, 실제 70억 파라미터급 모델 숫자로 ---
# 실제 ~70억 파라미터 트랜스포머를 참고한 설정: 레이어 32개, 어텐션 헤드 32개,
# head_dim=128 (따라서 d_model = 32*128 = 4096), fp16 가중치(값 하나당 2바이트).
n_layers, n_heads, head_dim, dtype_bytes = 32, 32, 128, 2

def kv_cache_bytes(seq_len, n_kv_heads, batch_size=1):
    # 레이어당 캐시 shape: (batch, n_kv_heads, seq_len, head_dim), K와 V 각각 하나씩
    per_token_per_layer = 2 * n_kv_heads * head_dim * dtype_bytes  # K와 V를 저장하므로 x2
    return batch_size * n_layers * seq_len * per_token_per_layer

print(f"{'seq_len':>8}  {'MHA (kv헤드 32개)':>20}  {'GQA (kv헤드 8개)':>18}  {'MQA (kv헤드 1개)':>16}  감소율")
for seq_len in [128, 1024, 8192, 32768]:
    mha = kv_cache_bytes(seq_len, n_kv_heads=32)  # 표준 멀티헤드 어텐션
    gqa = kv_cache_bytes(seq_len, n_kv_heads=8)   # 그룹 쿼리 어텐션: kv헤드 8개를 쿼리헤드 32개가 공유
    mqa = kv_cache_bytes(seq_len, n_kv_heads=1)   # 멀티 쿼리 어텐션: kv헤드 1개를 공유
    print(f"{seq_len:>8}  {mha/1e6:>17.1f} MB  {gqa/1e6:>15.1f} MB  {mqa/1e6:>13.1f} MB  GQA로 {mha/gqa:.1f}배 감소")

print("\n한 번의 생성 과정에서 KV 캐시 증가량 (MHA, batch=1):")
for n_generated in [1, 10, 100, 1000]:
    b = kv_cache_bytes(n_generated, n_kv_heads=32)
    print(f"  {n_generated:>4}토큰 후: {b/1e6:8.2f} MB")
# 검증됨: seq_len=32768에서 MHA 캐시만으로도 요청 1건당 ~17.2GB -- 모델 자체 가중치
# 크기와 맞먹는 수준. 이것이 바로 GQA/MQA(여기서는 kv헤드를 32개에서 8개로 줄여
# 정확히 4배 감소)가 대규모로 긴 컨텍스트 요청을 서빙할 때 중요한 이유.

## Day 2: 임베딩 모델과 재정렬

바이 인코더 실패 양상을 보여주는 TF-IDF 기반 데모(순수하게 어휘적이라 구조적 약점을
깔끔하게 드러냄), 실제 신경망 바이 인코더/크로스 인코더를 위한 실제 `sentence-transformers`
API(이 환경에서는 실행하지 않음), 그리고 장난감 순위 검색 결과 집합에 대해 처음부터
구현한 Recall@k / MRR 평가.

In [ ]:
# --- "독립적으로 인코딩한 뒤 벡터를 비교"의 실행 가능한 대체물로서의 TF-IDF ---
# TF-IDF는 바이 인코더와 같은 *형태*의 계산(각 텍스트를 독자적으로 고정 벡터로 인코딩한
# 뒤 코사인 유사도로 비교)을 하지만, 순수 어휘 특징만 쓰고 학습된 동의어 개념이 전혀
# 없어서 실패 양상을 최대한 뚜렷하게 보여줌.
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

query = "how do I reset my account password"

docs = [
    # 실제로 관련 있지만 다르게 표현됨 -- 쿼리와 문자 그대로 겹치는 단어가 거의 없음
    "Forgot your login credentials? Use the 'Trouble signing in' link on the sign-in page to create a new one.",
    # 주제는 완전히 다름(스마트 도어락 설명서)이지만 'account', 'password', 'reset'을 그대로 재사용
    "The account password reset button on our smart lock clears a stored 4-digit code back to its factory default.",
]

vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform([query] + docs).toarray()  # shape: (3, vocab_size)
q_vec, d_vecs = tfidf[0], tfidf[1:]                          # q_vec: (vocab_size,); d_vecs: (2, vocab_size)

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9)

for i, doc in enumerate(docs):
    print(f"sim={cosine_sim(q_vec, d_vecs[i]):.4f}  {doc[:65]}...")
# 실제 검증된 출력: 실제로 관련 있는 패러프레이즈는 0.0000점(문자 그대로 겹치는 단어가
# 전혀 없음), 반면 주제가 다른 스마트 도어락 문서는 순전히 'account'/'password'/'reset'을
# 그대로 재사용했다는 이유만으로 0.2027점 -- 어려운 부정 예시가 충분하지 않은 채로
# 학습된 바이 인코더가 정도는 약하지만 여전히 보일 수 있는 것과 같은 실패 양상.

In [ ]:
# --- 실제 sentence-transformers API (현재 시그니처) -- 이 샌드박스에서는 실행하지 않음 ---
# 이 환경엔 torch/sentence-transformers가 설치돼 있지 않고 GPU도 없음; 올바른 최신
# 메서드 이름과 shape를 보여주기 위한 것으로, 위 셀의 실제 신경망 버전에 해당함.
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
query = "how do I reset my account password"
docs = [
    "Forgot your login credentials? Use the 'Trouble signing in' link on the sign-in page to create a new one.",
    "The account password reset button on our smart lock clears a stored 4-digit code back to its factory default.",
]

q_vec = bi_encoder.encode(query)   # shape: (384,), 이 모델 기준
d_vecs = bi_encoder.encode(docs)   # shape: (2, 384)
bi_scores = d_vecs @ q_vec / (np.linalg.norm(d_vecs, axis=1) * np.linalg.norm(q_vec))  # shape: (2,)

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
cross_scores = cross_encoder.predict([(query, d) for d in docs])  # shape: (2,) -- 쌍마다 스칼라 하나

In [ ]:
# --- 처음부터 구현한 Recall@k와 MRR, 장난감 순위 검색 결과 집합에 대해 ---
import numpy as np

relevant_doc = {"q1": "d7", "q2": "d3", "q3": "d9", "q4": "d1", "q5": "d5"}
ranked_results = {
    "q1": ["d2", "d7", "d4", "d9", "d1"],   # 정답 d7이 2위
    "q2": ["d3", "d8", "d1", "d2", "d4"],   # 정답 d3이 1위
    "q3": ["d2", "d4", "d1", "d8", "d6"],   # 정답 d9는 아예 없음 -> 누락
    "q4": ["d5", "d2", "d1", "d3", "d9"],   # 정답 d1이 3위
    "q5": ["d5", "d1", "d2", "d3", "d4"],   # 정답 d5가 1위
}

def recall_at_k(ranked, relevant, k):
    hits = sum(relevant[q] in docs[:k] for q, docs in ranked.items())
    return hits / len(ranked)

def mrr(ranked, relevant):
    reciprocal_ranks = []
    for q, docs in ranked.items():
        if relevant[q] in docs:
            reciprocal_ranks.append(1.0 / (docs.index(relevant[q]) + 1))  # 1부터 시작하는 순위
        else:
            reciprocal_ranks.append(0.0)  # 정답 문서를 아예 찾지 못함
    return np.mean(reciprocal_ranks)

for k in [1, 3, 5]:
    print(f"Recall@{k}: {recall_at_k(ranked_results, relevant_doc, k):.2f}")
print(f"MRR: {mrr(ranked_results, relevant_doc):.3f}")
# 검증됨: Recall@1=0.40, Recall@3=Recall@5=0.80 (q3의 정답 문서는 후보 목록에 아예
# 등장하지 않음 -- 어떤 재정렬로도 고칠 수 없는 1단계 검색 누락), MRR=0.567.

## Day 3: 오디오 AI — ASR과 TTS

처음부터 구현한 파형 -> STFT -> 로그-멜 스펙트로그램 파이프라인(Whisper를 포함해 모든
음성 트랜스포머가 내부적으로 실제로 수행하는 전처리 단계)을 오디오 라이브러리 없이
`numpy`/`scipy`만으로 만들고 실행했습니다. 이어서 실제 최신 Whisper API(이 환경에서는
실행하지 않음)와 OS 레벨 TTS 폴백을 다룹니다.

In [ ]:
# --- 처음부터 구현한 파형 -> STFT -> 로그-멜 스펙트로그램 ---
import numpy as np
from scipy.signal import stft

sample_rate = 16000  # Hz -- Whisper를 포함한 표준 ASR 입력 레이트
duration_s = 1.0
t = np.linspace(0, duration_s, int(sample_rate * duration_s), endpoint=False)  # shape: (16000,)

# "음성 비슷한" 합성 신호: 기본 주파수 + 배음 2개(노래하는 모음처럼),
# 음성 특유의 켜짐/꺼짐 포락선을 흉내 낸 진폭 변조, 약간의 노이즈를 더함.
np.random.seed(42)  # 노이즈 항을 고정해 출력이 재현되도록 함
f0 = 150  # Hz, 그럴듯한 음높이
signal = (1.0 * np.sin(2 * np.pi * f0 * t)
          + 0.5 * np.sin(2 * np.pi * 2 * f0 * t)
          + 0.3 * np.sin(2 * np.pi * 3 * f0 * t))
envelope = 0.5 * (1 + np.sin(2 * np.pi * 2 * t - np.pi / 2))
signal = signal * envelope + 0.02 * np.random.randn(len(t))  # shape: (16000,) float64, 원시 진폭 샘플
print("waveform shape:", signal.shape, signal.dtype)

# --- STFT: 파형 위로 25ms 윈도우를 10ms씩 슬라이딩 ---
win_length = int(0.025 * sample_rate)  # 400 샘플 = 25ms
hop_length = int(0.010 * sample_rate)  # 160 샘플 = 10ms
freqs, frame_times, Zxx = stft(signal, fs=sample_rate, nperseg=win_length, noverlap=win_length - hop_length)
magnitude = np.abs(Zxx)  # shape: (201, 101) -> (freq_bins, time_frames); 201 = win_length//2 + 1
print("STFT magnitude shape:", magnitude.shape, "-> (freq_bins, time_frames)")

In [ ]:
# --- 멜 필터뱅크: 표준 공식으로 직접 구현 (이 샌드박스엔 librosa 없음) ---
def hz_to_mel(f):
    return 2595 * np.log10(1 + f / 700)

def mel_to_hz(m):
    return 700 * (10 ** (m / 2595) - 1)

n_mels = 40
n_fft_bins = magnitude.shape[0]  # 201
mel_points = np.linspace(hz_to_mel(0), hz_to_mel(sample_rate / 2), n_mels + 2)  # shape: (42,) 멜 스케일 경계
bin_points = np.floor((n_fft_bins - 1) * 2 * mel_to_hz(mel_points) / sample_rate).astype(int)  # -> FFT bin 인덱스

filterbank = np.zeros((n_mels, n_fft_bins))  # shape: (40, 201) -- 삼각형 필터, 멜 대역마다 한 행
for m in range(1, n_mels + 1):
    left, center, right = bin_points[m - 1], bin_points[m], bin_points[m + 1]
    for k in range(left, center):
        if center != left:
            filterbank[m - 1, k] = (k - left) / (center - left)      # 삼각형의 상승 구간
    for k in range(center, right):
        if right != center:
            filterbank[m - 1, k] = (right - k) / (right - center)    # 삼각형의 하강 구간

with np.errstate(all="ignore"):  # 일부 플랫폼에서 0이 많은 행렬곱에 나오는 무해한 BLAS 경고
    mel_spec = filterbank @ magnitude  # shape: (40, 201) @ (201, 101) -> (40, 101)
log_mel_spec = np.log(mel_spec + 1e-6)  # shape: (40, 101) -- 로그 압축, 지각된 음량과 맞춰줌

print("log-mel spectrogram shape:", log_mel_spec.shape, "-> (n_mels, time_frames)")
print("value range:", round(float(log_mel_spec.min()), 2), "to", round(float(log_mel_spec.max()), 2))
# 검증됨: 1초 분량의 원시 오디오(숫자 16000개)가 (40, 101) 배열이 됨 -- 이것이 바로
# 음성 트랜스포머의 인코더가 실제로 소비하는 대상이며, 원시 샘플을 직접 쓰는 일은 없음.

In [ ]:
# --- 실제 Whisper API (현재 시그니처) -- 이 샌드박스에서는 실행하지 않음 ---
# 이 환경엔 whisper가 설치돼 있지 않고, 모델 가중치(수백 MB)와 추론 모두 이 환경에서는
# 현실적이지 않음; 올바른 최신 openai-whisper API를 보여주기 위한 것.
import whisper

model = whisper.load_model("base")

# 도메인 힌트가 없으면 "Postgres" 같은 드문 제품명이 엉뚱하게 전사될 수 있음(실제로
# 체계적으로 나타나는 실패 양상).
result_plain = model.transcribe("voicemail_042.wav")
print("plain:", result_plain["text"])

# 도메인 힌트를 주면 디코더가 예상 어휘 쪽으로 편향되어 디코딩됨
result_hinted = model.transcribe(
    "voicemail_042.wav",
    initial_prompt="Postgres, replica, failover, connection pool",
)
print("hinted:", result_hinted["text"])

In [ ]:
# --- OS 레벨 TTS 폴백: 추가 의존성 없이, 설정이 필요 없는 기본값으로 유용함 ---
import pyttsx3

engine = pyttsx3.init()
engine.say("The nightly backup completed with no errors.")
engine.runAndWait()

## Day 4: 에이전트를 위한 코드 실행 샌드박싱

호스팅형 샌드박스 SDK가 노출하는 일반적인 create/run/files/kill 생명주기(개략적인 예시 --
제공업체마다 메서드 이름은 다름)를 먼저 살펴본 뒤, 검증된 로컬 `subprocess` + `resource`
패턴을 다룹니다 -- CPU를 붙잡아 먹는 무한 루프를 만났을 때 무슨 일이 일어나는지, 그리고
이 플랫폼에서 어디가 조용히 실패하는지를 포함해서, 이것이 보안 경계가 아니라 방지턱일
뿐임을 명확히 표시합니다.

In [ ]:
# 호스팅형 샌드박스 SDK의 전형적인 생명주기를 보여주는 개략적인 예시.
# 예시일 뿐 -- 정확한 메서드 이름은 제공업체마다 다르지만, create -> run -> 파일을
# 안팎으로 옮김 -> 종료, 그리고 명시적인 네트워크 토글이라는 구조는 거의 보편적임.

sandbox = Sandbox.create(timeout=60, network_access=False)

output = sandbox.run_code("import statistics; print(statistics.mean([3, 7, 9, 12]))")

sandbox.files.write("/tmp/report.csv", csv_bytes)
sandbox.commands.run("pip install pandas")

sandbox.kill()

In [ ]:
# --- 검증됨: subprocess + RLIMIT_CPU가 실제로 폭주하는 CPU 루프를 멈추게 함 ---
import resource
import subprocess
import time

def limit_cpu():
    # preexec_fn은 자식 프로세스에서, fork() 이후 exec() 이전에 실행됨 -- 이 제한은
    # 서브프로세스에만 적용되며, 이 노트북을 실행 중인 부모 프로세스에는 적용되지 않음.
    resource.setrlimit(resource.RLIMIT_CPU, (2, 2))  # 2 CPU초, (soft, hard)

# 케이스 1: 정상적으로 동작하는 코드는 제한 안에서 정상적으로 실행되고 반환됨.
r1 = subprocess.run(["python3", "-c", "print(sum(range(1000)))"],
                     timeout=10, preexec_fn=limit_cpu, capture_output=True, text=True)
print("good_code:", r1.returncode, repr(r1.stdout.strip()))

# 케이스 2: CPU를 붙잡아 먹는 무한 루프 -- OS가 2 CPU초를 소모하는 즉시 SIGKILL해야 하며,
# 이는 외부의 timeout=10이 발동하기 훨씬 전이어야 함.
start = time.time()
r2 = subprocess.run(["python3", "-c", "while True: pass"],
                     timeout=10, preexec_fn=limit_cpu, capture_output=True, text=True)
elapsed = time.time() - start
print(f"bad_code (infinite loop): returncode={r2.returncode} elapsed={elapsed:.2f}s")
# 실제 검증된 출력: returncode=-24 (SIGXCPU로 종료됨), 약 2.0초 후. 이 부분은
# 실제로 잘 동작하며 플랫폼에 관계없이 동일함.

In [ ]:
# --- 검증됨: RLIMIT_AS(메모리 제한)는 플랫폼을 넘나들며 신뢰성 있게 강제되지 않음 ---
# 바로 이것이 "subprocess + resource 제한"을 절대 진짜 보안 경계로 취급해서는 안 되는
# 이유임 -- 균일하게 동작해야 할 것처럼 보이는 부분조차 특정 플랫폼에서는 조용히
# 실패할 수 있음.
import resource

try:
    resource.setrlimit(resource.RLIMIT_AS, (256 * 1024 * 1024,) * 2)  # 시도: 256MB로 제한
    print("RLIMIT_AS set successfully")
except ValueError as e:
    print("RLIMIT_AS failed on this platform:", e)
# macOS에서 실제 검증된 출력: "ValueError: current limit exceeds maximum limit" --
# 메모리 제한이 여기서는 조용히 적용되지 않는 반면, 위의 CPU 제한은 적용됨.